In [1]:
import re
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import DoubleType, DateType

In [2]:
%run ../ingestion/functions.ipynb

In [3]:
def camel_to_snake(name):
    return re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()

In [4]:
# --- Cleanse accounts ---
accounts_raw = read_delta("data/raw/accounts")
accounts_cleansed = accounts_raw.withColumn("balance", col("balance").cast(DoubleType()))
accounts_cleansed = accounts_cleansed.select(
    [col(c).alias(camel_to_snake(c)) for c in accounts_cleansed.columns]
)
accounts_cleansed = accounts_cleansed.withColumn("processed_at", current_timestamp())
write_delta(accounts_cleansed, "data/cleansed/accounts")
print(f"cleansed/accounts: {accounts_cleansed.count()} rows written")
accounts_cleansed.printSchema()

cleansed/accounts: 2 rows written
root
 |-- balance: double (nullable = true)
 |-- bank_data: struct (nullable = true)
 |    |-- automaticallyInvestedBalance: double (nullable = true)
 |    |-- closingBalance: double (nullable = true)
 |    |-- overdraftContractedLimit: long (nullable = true)
 |    |-- overdraftUsedLimit: long (nullable = true)
 |    |-- transferNumber: string (nullable = true)
 |    |-- unarrangedOverdraftAmount: long (nullable = true)
 |-- created_at: string (nullable = true)
 |-- credit_data: struct (nullable = true)
 |    |-- additionalCards: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- availableCreditLimit: double (nullable = true)
 |    |-- balanceCloseDate: string (nullable = true)
 |    |-- balanceDueDate: string (nullable = true)
 |    |-- balanceForeignCurrency: string (nullable = true)
 |    |-- brand: string (nullable = true)
 |    |-- creditLimit: long (nullable = true)
 |    |-- disaggregatedCreditLimits: array (n

In [5]:
# --- Cleanse transactions ---
transactions_raw = read_delta("data/raw/transactions")
transactions_cleansed = transactions_raw.withColumn("amount", col("amount").cast(DoubleType()))
transactions_cleansed = transactions_cleansed.withColumn("date", col("date").cast(DateType()))
transactions_cleansed = transactions_cleansed.select(
    [col(c).alias(camel_to_snake(c)) for c in transactions_cleansed.columns]
)
transactions_cleansed = transactions_cleansed.withColumn("processed_at", current_timestamp())
write_delta(transactions_cleansed, "data/cleansed/transactions")
print(f"cleansed/transactions: {transactions_cleansed.count()} rows written")
transactions_cleansed.printSchema()

cleansed/transactions: 532 rows written
root
 |-- account_id: string (nullable = true)
 |-- acquirer_data: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- amount_in_account_currency: double (nullable = true)
 |-- balance: string (nullable = true)
 |-- category: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- credit_card_metadata: struct (nullable = true)
 |    |-- billId: string (nullable = true)
 |    |-- cardNumber: string (nullable = true)
 |    |-- installmentNumber: long (nullable = true)
 |    |-- payeeMCC: long (nullable = true)
 |    |-- purchaseDate: string (nullable = true)
 |    |-- totalInstallments: long (nullable = true)
 |-- currency_code: string (nullable = true)
 |-- date: date (nullable = true)
 |-- description: string (nullable = true)
 |-- description_raw: string (nullable = true)
 |-- id: string (nullable = true)
 |-- merchant: struct (nullable = true)
 |    |-- businessName: str

In [6]:
delta_to_csv("data/cleansed/transactions", "data/cleansed/csv/transactions.csv")
delta_to_csv("data/cleansed/accounts", "data/cleansed/csv/accounts.csv" )